# 02 — Trend Signal Construction

This notebook converts monthly ETF prices into cross-asset trend signals for the portfolio backtest. The signal is timestamped at the month-end when it becomes observable; the trading lag is applied in the notebook 04.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
SIGNAL_DIR = Path("results") / "trend_signal"
SIGNAL_DIR.mkdir(parents=True, exist_ok=True)
LOOKBACK_MONTHS = 12
WEIGHT_LAG_MONTHS = 1

prices = pd.read_csv(DATA_DIR / "prices.csv", index_col=0, parse_dates=True)
returns = pd.read_csv(DATA_DIR / "returns.csv", index_col=0, parse_dates=True)

default_layout = dict(
    template="plotly_white",
    height=500,
    margin=dict(t=70, l=60, r=40, b=60),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)


In [3]:
def compute_signal_summary(signal, raw_trend):
    """Summarize signal strength, switching activity, and underlying trend behavior."""
    common_index = signal.index.intersection(raw_trend.index)
    signal_block = signal.loc[common_index]
    trend_block = raw_trend.loc[common_index]

    valid_change = signal_block.notna() & signal_block.shift(1).notna()
    signal_changes = (signal_block.diff().ne(0) & valid_change).sum()

    summary = pd.DataFrame({
        "Average Signal (%)": signal_block.mean() * 100,
        "Signal Changes": signal_changes,
        "Average Trend Return (%)": trend_block.mean() * 100,
        "Trend Return Volatility (%)": trend_block.std() * 100,
    })

    return summary.sort_values("Average Signal (%)", ascending=False).round(2)


## 1. Monthly returns and 12-month trend signal

Daily adjusted prices are converted to month-end prices and monthly returns. If the latest download includes an incomplete current month, that row is dropped so that a partial month is not treated as a full monthly observation.

The first signal is a 12-month absolute trend rule: an asset receives a score of 1 when its trailing 12-month price return is positive and 0 when it is negative. Missing lookback periods are left as missing rather than being treated as negative signals.


In [ ]:
monthly_prices = prices.resample("ME").last()

current_month = pd.Timestamp.today().to_period("M")
if monthly_prices.index[-1].to_period("M") == current_month:
    monthly_prices = monthly_prices.iloc[:-1]

monthly_returns = monthly_prices.pct_change().dropna(how="any")

monthly_prices.tail()

Monthly price data shape: (230, 11)


,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-01-31,690.0853,100.7400,59.1000,94.4057,85.5127,108.4968,79.1176,444.9500,24.4300,89.8396,81.7867
2026-02-28,684.1216,105.3800,62.5800,96.7375,89.4751,109.9918,79.1166,483.7500,25.1000,94.6779,82.2217
2026-03-31,650.3400,97.1300,56.7900,94.4930,85.6903,107.7119,78.3622,430.2900,28.9500,88.7000,81.8381
2026-04-30,718.6600,102.3200,63.9900,94.3509,84.9708,108.0155,79.5538,423.6600,31.1000,96.3300,81.9942
2026-05-31,756.4800,104.8000,68.6000,94.3330,85.4240,108.9470,79.9010,417.1200,29.4800,95.7000,82.0570


In [ ]:
trend_12m = monthly_prices.pct_change(LOOKBACK_MONTHS)
signal_12m = trend_12m.gt(0).astype(float).where(trend_12m.notna())

signal_12m.tail()

,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-01-31,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-02-28,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-03-31,1.0000,1.0000,1.0000,1.0000,0.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-04-30,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-05-31,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


## 2. 12-month trend diagnostics

The diagnostic charts focus on a representative subset of assets to keep the output readable while still covering equities, duration, gold, and commodities. The in-sample window ends in 2017; later notebooks use the post-2017 period for portfolio evaluation.


In [ ]:
IS_END = "2017-12-31"
plot_start = "2007-01-01"
plot_end = IS_END

assets_to_plot = ["SPY", "EFA", "IEF", "TLT", "GLD", "DBC"]
trend_12m_plot = trend_12m.loc[plot_start:plot_end, assets_to_plot]
signal_12m_plot = signal_12m.loc[plot_start:plot_end, assets_to_plot]

In [7]:
fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(go.Scatter(x=trend_12m_plot.index, y=trend_12m_plot[asset], mode="lines", name=asset))

fig.add_hline(y=0, line_dash="dash")
fig.update_layout(
    **default_layout,
    title="12-Month Trend Returns",
    xaxis_title="Date",
    yaxis_title="Trailing 12-Month Return",
)
fig.update_yaxes(tickformat=".0%")
fig.show()


The trend-return chart is a regime check rather than a performance result. The rule should switch off during sustained weakness in risk assets and behave differently across defensive and inflation-sensitive assets.


In [8]:
fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(
        go.Scatter(
            x=signal_12m_plot.index,
            y=signal_12m_plot[asset],
            mode="lines",
            name=asset,
            line_shape="hv",
        )
    )

fig.update_layout(
    **default_layout,
    title="12-Month Trend Signals",
    xaxis_title="Date",
    yaxis_title="Signal: 1 = positive trend, 0 = negative trend",
)
fig.show()


### In-sample and full-sample signal summaries

The average signal measures how often, or how strongly, an asset is eligible for risk allocation. Signal changes indicate potential turnover pressure before portfolio weighting is applied.


In [ ]:
signal_summary_12m_is = compute_signal_summary(
    signal=signal_12m.loc[:IS_END],
    raw_trend=trend_12m.loc[:IS_END],
)

signal_summary_12m_is

,Average Signal (%),Signal Changes,Average Trend Return (%),Trend Return Volatility (%)
SHY,99.1500,2,1.5400,1.8600
LQD,84.6200,10,5.8800,6.8000
SPY,81.2000,7,9.5000,18.0300
VNQ,79.4900,7,10.2600,26.4900
IEF,78.6300,8,5.0700,5.7200
HYG,78.6300,12,6.4800,12.0900
TLT,72.6500,12,7.4100,12.5300
GLD,58.1200,22,6.2900,17.4700
EEM,57.2600,14,4.2900,25.6100
EFA,55.5600,11,3.4900,20.3100


In [10]:
signal_summary_12m_full = compute_signal_summary(
    signal=signal_12m,
    raw_trend=trend_12m,
)

signal_summary_12m_full


,Average Signal (%),Signal Changes,Average Trend Return (%),Trend Return Volatility (%)
SHY,85.3200,14,1.6900,2.4000
SPY,83.4900,13,12.2500,16.5800
HYG,80.2800,22,5.5600,9.9300
LQD,78.4400,20,4.4900,7.8200
VNQ,71.1000,23,8.4400,22.3100
IEF,66.0600,16,3.3400,6.9300
GLD,65.6000,32,10.8600,18.9400
EFA,62.8400,21,5.9500,18.1100
EEM,60.0900,24,5.7500,22.9700
TLT,59.1700,26,3.8400,14.0500


## 3. Multi-horizon trend signal

A single 12-month rule is transparent but dependent on one lookback window. The final signal averages four binary trend rules based on 3-month, 6-month, 9-month and 12-month returns. This keeps the rule simple while reducing dependence on one horizon.

Signal interpretation: 0.00 = no trend support, 0.25 = weak support, 0.50 = moderate support, 0.75 = strong support and 1.00 = full support across all four horizons.


In [11]:
trend_3m = monthly_prices.pct_change(3)
trend_6m = monthly_prices.pct_change(6)
trend_9m = monthly_prices.pct_change(9)
trend_12m = monthly_prices.pct_change(12)
trend_multi = (trend_3m + trend_6m + trend_9m + trend_12m) / 4

signal_3m = trend_3m.gt(0).astype(float).where(trend_3m.notna())
signal_6m = trend_6m.gt(0).astype(float).where(trend_6m.notna())
signal_9m = trend_9m.gt(0).astype(float).where(trend_9m.notna())
signal_12m = trend_12m.gt(0).astype(float).where(trend_12m.notna())
signal_multi = (signal_3m + signal_6m + signal_9m + signal_12m) / 4


In [12]:
signal_multi_plot = signal_multi.loc[plot_start:plot_end, assets_to_plot]

fig = go.Figure()
for asset in assets_to_plot:
    fig.add_trace(
        go.Scatter(
            x=signal_multi_plot.index,
            y=signal_multi_plot[asset],
            mode="lines",
            name=asset,
            line_shape="hv",
        )
    )

fig.update_layout(
    **default_layout,
    title="Multi-Horizon Trend Signal",
    xaxis_title="Date",
    yaxis_title="Signal score",
)
fig.show()


In [13]:
signal_summary_multi = compute_signal_summary(
    signal=signal_multi,
    raw_trend=trend_multi,
)

signal_summary_multi


,Average Signal (%),Signal Changes,Average Trend Return (%),Trend Return Volatility (%)
SHY,80.0500,66,1.0200,1.4700
SPY,79.1300,68,7.6700,10.9300
HYG,76.9500,74,3.4900,6.6300
LQD,70.6400,90,2.7900,5.3300
VNQ,68.6900,100,5.4100,14.4400
GLD,64.3300,107,6.6500,12.3000
EFA,63.6500,89,3.8800,12.3400
IEF,63.4200,96,2.0100,4.6600
EEM,58.9400,93,3.7200,15.3000
TLT,56.4200,92,2.3500,9.6900


## 4. Final signal file for portfolio construction

The multi-horizon signal is selected for the portfolio stage because it is simple, interpretable, and less dependent on a single lookback window. The backtest notebook will apply a one-month trading lag when converting these month-end signals into returns.


In [14]:
common_index = monthly_returns.index.intersection(signal_multi.index)

final_signal = signal_multi.loc[common_index]
monthly_returns_aligned = monthly_returns.loc[common_index]

valid_dates = final_signal.dropna(how="any").index
final_signal = final_signal.loc[valid_dates]
monthly_returns_aligned = monthly_returns_aligned.loc[valid_dates]

print("Final signal shape:", final_signal.shape)
print("Aligned monthly returns shape:", monthly_returns_aligned.shape)
display(final_signal.tail())


Final signal shape: (218, 11)
Aligned monthly returns shape: (218, 11)


,SPY,EFA,EEM,IEF,TLT,LQD,HYG,GLD,DBC,VNQ,SHY
Date,,,,,,,,,,,
2026-01-31,1.0000,1.0000,1.0000,0.7500,0.7500,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-02-28,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
2026-03-31,0.5000,1.0000,1.0000,0.7500,0.5000,0.5000,0.7500,1.0000,1.0000,0.7500,1.0000
2026-04-30,1.0000,1.0000,1.0000,0.5000,0.5000,0.7500,1.0000,0.7500,1.0000,1.0000,1.0000
2026-05-31,1.0000,0.7500,1.0000,0.5000,0.5000,0.7500,1.0000,0.7500,1.0000,1.0000,0.7500


In [15]:
final_signal.to_csv(SIGNAL_DIR / "final_signal.csv")
monthly_returns_aligned.to_csv(SIGNAL_DIR / "monthly_returns_aligned.csv")

print("Saved trend-signal files to:", SIGNAL_DIR.resolve())


Saved trend-signal files to: C:\Users\mmodr\Desktop\project-trend-invvol-strategy\results\trend_signal
